# C1 · Breaking the KNN with noise columns

**Type**: Break the model · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Add to a two-dimensional problem a growing number of pure noise columns and measure how a KNN collapses while a logistic and a random forest hold up. Measure the **concentration of the distances** (the nearest neighbor
becomes almost as far as the farthest) and repair with feature selection inside the pipeline.

## Dataset

`generate(d)` (given): 3000 points of `make_moons(noise=0.25)` with `d` columns of Gaussian noise added, 50/50 split in `X_tr, X_te, y_tr, y_te`. The two informative columns are always the first two.

## Educational purpose

See with numbers why KNN is the model most exposed to the curse of dimensionality: every useless column adds a random component to the distance, and with enough columns the "true" neighbors are no longer the nearest ones.
The same does not happen to models that select the features (trees) or ignore them (regularized logistic).

## Task

1. For `d` in `[0, 2, 5, 10, 20, 50, 100, 200]`: generate the data and compute on the test set the accuracy of `KNeighborsClassifier(5)`, of `LogisticRegression(max_iter=1000)` and of `RandomForestClassifier(100, random_state=0, n_jobs=-1)`, and `concentration` (given: `concentration(X_te)`, ratio between the mean distance to the nearest neighbor and the mean distance to the farthest on a sample). Save everything in `table` (index `d`, columns `knn`, `lr`, `rf`, `concentration`).
2. Save `collapse_d`: the first `d` at which the KNN accuracy drops **below 0.65**.
3. Save `loss`: dictionary `{'knn': ..., 'lr': ..., 'rf': ...}` with `accuracy(d=0) − accuracy(d=200)` for each model.
4. **Repair:** for `d = 100` train a `Pipeline` `SelectKBest(f_classif, k=2)` + `KNeighborsClassifier(5)` and save `selection_acc` (test accuracy) and `chosen_columns` (indices of the selected columns: `get_support().nonzero()[0]`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline


def generate(d, seed=0):
    X, y = make_moons(3000, noise=0.25, random_state=0)
    noise = np.random.default_rng(seed).normal(size=(len(X), d))
    return train_test_split(np.c_[X, noise], y, test_size=0.5, random_state=0)


def concentration(X, n=300):
    D = np.sqrt(((X[:n, None, :] - X[None, n:2 * n, :]) ** 2).sum(-1))       # distances between two different groups of points
    return D.min(1).mean() / D.max(1).mean()

## Your work

In [ ]:
rows = {}
for d in [0, 2, 5, 10, 20, 50, 100, 200]:
    X_tr, X_te, y_tr, y_te = generate(d)
    rows[d] = ...
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
collapse_d = None
loss = None

In [ ]:
X_tr, X_te, y_tr, y_te = generate(100)
selection_acc = None
chosen_columns = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Row: `{"knn": KNeighborsClassifier(5).fit(X_tr, y_tr).score(X_te, y_te), "lr": ..., "rf": ..., "concentration": concentration(X_te)}`. Selection: `sel = make_pipeline(SelectKBest(f_classif, k=2), KNeighborsClassifier(5)).fit(X_tr, y_tr)`; `sel[0].get_support().nonzero()[0]`.'):
    check(list(table.columns) == ["knn", "lr", "rf", "concentration"] and list(table.index) == [0, 2, 5, 10, 20, 50, 100, 200], "table with the four columns for the eight values of d", "index d, columns knn, lr, rf, concentration")
    check(table.loc[0, "knn"] > 0.9 and table.loc[200, "knn"] < 0.62, "the KNN goes from over 0.9 to less than 0.62 (almost chance)", "test accuracy at d = 0 and d = 200")
    check(table.loc[200, "lr"] > 0.8 and table.loc[200, "rf"] > 0.85, "the logistic and the random forest stay above 0.8 and 0.85", "they don't use the distances in the full space")
    check(table["concentration"].is_monotonic_increasing and table.loc[0, "concentration"] < 0.05 and table.loc[200, "concentration"] > 0.7, "the nearest/farthest ratio goes from below 0.05 to above 0.7", "with noise the nearest neighbor is as far as the farthest")
    check(collapse_d in (50, 100), "the KNN drops below 0.65 with 50-100 noise columns", "first d with knn < 0.65")
    check(loss["knn"] > 0.3 and loss["lr"] < 0.06 and loss["rf"] < 0.06, "loss: KNN over 0.3, the others below 0.06", "accuracy(0) - accuracy(200)")
    check(selection_acc > 0.92 and sorted(chosen_columns.tolist()) == [0, 1], "with the selection the accuracy goes back to 0.92 or more and the chosen columns are the first two", "SelectKBest inside the pipeline")
done()

---
The commented solution is in `solutions/C_break_the_model/C1_curse_of_dimensionality.ipynb`. Open it after you pass the check, or after you have really tried.